# Model Alignment Lab

## Setup

In [ ]:
!wget -O lab.zip https://github.com/DerrickJ1612/model-alignment-lab/archive/refs/heads/workshop_ucsd.zip
!unzip -q lab.zip
%cd model-alignment-lab-workshop_ucsd
!pip install -q -r requirements-colab.txt
!pip uninstall -y torchao

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

from model_alignment_lab.utils.helpers import generate_response, format_example
from model_alignment_lab.evaluation.eval import log_parser, evaluate_tutor_schema_df

from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
## Path Setup
root = Path.cwd()
output_dir = root/"outputs"
datasets_dir = root/"datasets"/"structured_json"
root

In [ ]:
print(root.is_dir())
print(output_dir.is_dir())
print(datasets_dir.is_dir())

## Base Model Preparation

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32
)

model = model.to(device)
model.eval()

In [ ]:
prompt = """Return ONLY valid JSON.

You are a helpful personal electromagnetics tutor.
Provide structured reasoning and educational guidance.

Problem:
A conducting triangular loop is placed near a long straight wire carrying i(t)=I sin(wt). Determine the RMS induced voltage."""

response = generate_response(model, tokenizer, prompt)
print(response)

In [ ]:
result = generate_response(
    model,
    tokenizer,
    prompt,
    max_new_tokens=30,
    benchmark=True
)
print(f"Total Time: {np.round(result['total_time_s'],2)}")
print(f"Generated Tokens: {np.round(result['generated_tokens'],2)}")
print(f"Tokens per second: {np.round(result['tokens_per_second'],2)}")

## Model Alignment

### Dataset Preparation

In [ ]:
train_path = datasets_dir.joinpath("TRAIN_em_tutor_chat.jsonl")
test_path = datasets_dir.joinpath("VAL_em_tutor_chat.jsonl")
val_path = datasets_dir.joinpath("TEST_em_tutor_chat.jsonl")

In [ ]:
print(train_path.is_file())
print(test_path.is_file())
print(val_path.is_file())

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json",
                             data_files={
                                 "train": str(train_path)
                             }
                            )["train"]
test_dataset = load_dataset("json",
                            data_files={
                                "test": str(test_path)
                            }
                           )["test"]
val_dataset = load_dataset("json",
                           data_files={
                               "val":str(val_path)
                           }
                          )["val"]

In [ ]:
train_dataset

In [ ]:
train_dataset["messages"][0]

In [ ]:
train_dataset = train_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
val_dataset = val_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})
test_dataset = test_dataset.map(format_example, fn_kwargs={"tokenizer":tokenizer})

In [ ]:
train_dataset

In [ ]:
print(train_dataset["text"][0])

### LoRA Setup

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","v_proj"]
)

In [ ]:
model = get_peft_model(model, peft_config).to(model.device)
model.train()

In [ ]:
model.print_trainable_parameters()

### LoRA Training

In [ ]:
training_args = SFTConfig(
    output_dir=str(output_dir/"smollm-em-tutor-lora"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    use_cpu=False,
    bf16=False,
    fp16=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
final_path = str(output_dir/"smollm-em-tutor-lora")
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")

### Model Evaluation

In [ ]:
trainer.state.log_history

In [ ]:
train_epochs, train_losses, eval_epochs, eval_losses = log_parser(trainer)

figure, ax = plt.subplots()

ax.plot(train_epochs, train_losses, marker="o", label="Train Loss")
ax.plot(eval_epochs, eval_losses, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
print(prompt)

In [ ]:
response = generate_response(model, tokenizer, prompt)
response

In [ ]:
df = evaluate_tutor_schema_df(model, tokenizer, test_dataset)
df.head()

In [ ]:
df.problem_type_match.value_counts()

In [ ]:
df.difficulty_match.value_counts()

In [ ]:
df.final_answer_match.value_counts()

### LoRA Training v2

In [ ]:
trainer.args.num_train_epochs = 3
trainer.train()

### Model Evaluation v2

In [ ]:
train_epochs, train_losses, eval_epochs, eval_losses = log_parser(trainer)

figure, ax = plt.subplots()

ax.plot(train_epochs, train_losses, marker="o", label="Train Loss")
ax.plot(eval_epochs, eval_losses, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
print(prompt)

In [ ]:
generate_response(model, tokenizer, prompt)

In [ ]:
df_2 = evaluate_tutor_schema_df(model, tokenizer, test_dataset)
df_2.head()

In [ ]:
df_2.problem_type_match.value_counts()

In [ ]:
df_2.difficulty_match.value_counts()

In [ ]:
df_2.final_answer_match.value_counts()

In [ ]:
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v2")

### Full Circle


In [ ]:
prompt = """Return ONLY valid JSON.

You are a helpful personal electromagnetics tutor.
Provide structured reasoning and educational guidance.

Problem:
A uniformly charged sphere has radius R and charge density ρ. Find the electric field inside and outside the sphere."""

structured_output = generate_response(model, tokenizer, prompt)
print(structured_output)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)
print(base_model)


In [ ]:
prompt_final = f"""
You are a helpful electromagnetics tutor.

Given the structured analysis below, explain the problem
to a student in a natural educational way.

Structured analysis:
{structured_output}

Tutor explanation:
"""
response = generate_response(base_model, tokenizer, prompt_final, max_new_tokens=500)
print(response)

### Rank Reduction

In [ ]:
peft_config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","v_proj"]
)

model = get_peft_model(base_model, peft_config).to(base_model.device)
model.train()

In [ ]:
training_args = SFTConfig(
    output_dir=str(output_dir/"smollm-em-tutor-lora-r4"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="text",
    use_cpu=False,
    bf16=False,
    fp16=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
final_path = str(output_dir/"smollm-em-tutor-lora-r4")
trainer.model.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")
tokenizer.save_pretrained(f"{final_path}/{ts}_final_adapter_v1")

In [ ]:
trainer.state.log_history

In [ ]:
train_epochs, train_losses, eval_epochs, eval_losses = log_parser(trainer)

figure, ax = plt.subplots()

ax.plot(train_epochs, train_losses, marker="o", label="Train Loss")
ax.plot(eval_epochs, eval_losses, marker="x", label="Eval Loss")

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Evaluation Loss")

ax.legend()
ax.grid()

plt.show()

In [ ]:
print(prompt)

In [ ]:
response = generate_response(model, tokenizer, prompt)
response

In [ ]:
df = evaluate_tutor_schema_df(model, tokenizer, test_dataset)
df.head()

In [ ]:
df.problem_type_match.value_counts()

In [ ]:
df.difficulty_match.value_counts()

In [ ]:
df.final_answer_match.value_counts()

## Next Steps

### Experiment 1: Scale to Larger Base Models
Repeat the same fine-tuning workflow using larger models to observe differences in alignment quality and generalization.

**Suggested models:**
- `Qwen/Qwen2.5-3B-Instruct`
- `Qwen/Qwen2.5-7B-Instruct`
- `mistralai/Mistral-7B-v0.1`

**What to observe:**
- Does the model require fewer examples to align?
- Does it better preserve general capabilities?
- How do latency and memory usage scale?

---

### Experiment 2: Increase LoRA Rank
Try increasing the LoRA rank (`r`) to give the model more capacity to adapt.

**Suggested values:**
- `r = 16`
- `r = 32`
- `r = 64`

**Key idea:**
Higher rank allows the model to **override strong base model priors**, which is especially useful for structured outputs like JSON.

**Tradeoffs:**
- Increased memory usage
- Potential overfitting if dataset is small

---

### Experiment 3: Tune LoRA Hyperparameters
Experiment with additional LoRA settings to improve performance.

**Parameters to explore:**
- `lora_alpha` (scaling factor)
- `lora_dropout` (regularization)
- Target modules (e.g., attention layers only vs all linear layers)

**Ideas:**
- Increase `lora_alpha` to amplify learned updates
- Reduce dropout if the model is underfitting
- Apply LoRA selectively to different parts of the model

---

### Experiment 4: Use QLoRA for Larger Models
When working with larger models (e.g., 7B+), use QLoRA to reduce memory requirements.

**Benefits:**
- Enables fine-tuning with limited GPU memory
- Maintains strong performance with 4-bit quantization

**When to use:**
- GPU memory becomes a bottleneck
- Scaling beyond smaller models is required